In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import re

In [2]:
YEAR = 2022
EXPORT_FILENAME = f'data/{YEAR}/kenpom_{YEAR}.csv'

url = 'http://kenpom.com/index.php'

# Can also find archived version
# url = "https://web.archive.org/web/20220317101252/https://kenpom.com/"

f = requests.get(url)
soup = BeautifulSoup(f.text, "lxml")

In [3]:
COLUMNS = [
    'Rank', 'Team', 'Conference', 'W-L', 'AdjustEM',
    'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
    'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
    'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
    'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank'
]

In [4]:
df = pd.DataFrame(
    [
        [tr.text for tr in row.find_all('td')]
        for row in soup.find('table', {'id': 'ratings-table'}).find_all(class_=r"tourney")
    ],
    columns=COLUMNS
)

In [5]:
df.shape

(64, 21)

In [6]:
# Returns true if given string is a number and a valid seed number (1-16)
def is_valid_seed(x):
    return str(x).replace(' ', '').isdigit() and int(x) > 0 and int(x) <= 16


def parse_name(row):
    if is_valid_seed(row['Team'][-2:]):
        row['Seed'] = row['Team'][-2:].strip()
        row['Team'] = row['Team'][:-2].strip()
    else:
        row['Seed'] = np.NaN
        row['Team'] = row['Team']
    return row


# Parse out seed/team
df = df.apply(parse_name, axis=1)

In [7]:
# Split W-L column into wins and losses
df['Wins'] = df['W-L'].apply(lambda x: int(re.sub('-.*', '', x)))
df['Losses'] = df['W-L'].apply(lambda x: int(re.sub('.*-', '', x)))
df.drop('W-L', inplace=True, axis=1)

# Reorder columns just cause I'm OCD
df = df[['Rank', 'Seed', 'Team', 'Conference', 'Wins', 'Losses', 'AdjustEM',
         'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
         'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
         'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
         'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank']]

# Drop non tournament teams
df = df.dropna()

In [8]:
df

,Rank,Seed,Team,Conference,Wins,Losses,AdjustEM,AdjustO,AdjustO Rank,AdjustD,...,Luck,Luck Rank,SOS Pyth,SOS Pyth Rank,SOS OppO,SOS OppO Rank,SOS OppD,SOS OppD Rank,NCSOS Pyth,NCSOS Pyth Rank
0,1,1,Gonzaga,WCC,26,3,+32.95,121.8,1,88.9,...,-.038,270,+3.72,100,104.4,112,100.7,92,-2.55,243
1,2,1,Arizona,P12,31,3,+27.24,119.6,5,92.4,...,+.046,76,+6.33,69,106.0,75,99.7,56,-0.33,175
2,3,2,Kentucky,SEC,26,7,+26.63,120.1,4,93.5,...,-.032,252,+9.10,29,107.0,51,97.9,17,-2.79,250
3,4,1,Baylor,B12,26,6,+26.52,117.9,9,91.4,...,+.019,140,+10.89,10,107.5,33,96.6,5,-1.43,206
4,5,5,Houston,Amer,29,5,+26.51,117.3,10,90.8,...,-.007,194,+4.28,94,104.5,108,100.3,76,+0.36,152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,149,15,Cal St. Fullerton,BW,21,10,+1.91,104.6,145,102.6,...,+.038,94,-3.15,224,100.4,282,103.5,178,-3.23,262
60,150,16,Georgia St.,SB,18,10,+1.66,101.5,202,99.8,...,+.062,54,-1.51,184,100.9,252,102.4,129,+2.08,110
61,168,16,Norfolk St.,MEAC,24,6,+0.05,102.4,190,102.4,...,+.095,23,-10.72,358,95.1,358,105.8,281,-4.26,285
62,175,16,Wright St.,Horz,22,13,-0.34,106.6,107,106.9,...,+.005,165,-7.15,326,100.1,298,107.3,338,-0.15,165


In [9]:
df.to_csv(EXPORT_FILENAME, index=False)

# Old Version
This doesn't work with NIT seeds

In [ ]:
table_html = soup.find_all('table', {'id': 'ratings-table'})

In [3]:
# Weird issue w/ <thead> in the html
# Prevents us from just using pd.read_html
# Let's find all the thead contents and just replace/remove them
# This allows us to easily put the table row data into a dataframe using pandas
thead = table_html[0].find_all('thead')

table = table_html[0]
for x in thead:
    table = str(table).replace(str(x), '')

In [4]:
#    table = "<table id='ratings-table'>%s</table>" % table
df = pd.read_html(table, converters={3: lambda x: str(x)})[0]

df.columns = COLUMNS